# MedGemma Glaucoma Expert — Colab Flask upload API with XAI heatmap

A user uploads one retinal image through a website; Colab runs prediction + explanation + occlusion sensitivity XAI heatmap for that exact image, then returns JSON.


In [ ]:
# Install dependencies
!pip install -U -q peft bitsandbytes accelerate safetensors flask flask-cors pyngrok matplotlib


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 155.7 MB/s eta 0:00:00


In [ ]:
import os
import glob
from pathlib import Path
import io
import base64

import torch
from torch.nn import functional as F
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

from huggingface_hub import login
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftModel

from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
from werkzeug.utils import secure_filename
import uuid


## Hugging Face login

MedGemma is gated. Run this and log in with a Hugging Face token that has access to `google/medgemma-1.5-4b-it`.

In [ ]:
login()


## Locate adapter folder

Upload or copy your saved LoRA adapter folder into Colab. The default path below is `/content/medgemma-glaucoma-expert`.

In [ ]:
# Put your saved LoRA adapter folder in Colab, for example:
# /content/medgemma-glaucoma-expert
#
# It must contain adapter_config.json, adapter_model.safetensors, and processor files.

ADAPTER_DIR = "/content/medgemma-glaucoma-expert"  # change only if your folder is elsewhere

def find_adapter_dir():
    candidates = [
        ADAPTER_DIR,
        "medgemma-glaucoma-expert",
        "/content/medgemma-glaucoma-expert",
        "/content/drive/MyDrive/medgemma-glaucoma-expert",
    ]

    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.is_dir() and (candidate / "adapter_config.json").exists():
            return str(candidate)

    raise FileNotFoundError(
        "Could not find adapter_config.json. Upload/copy your saved adapter folder to "
        "/content/medgemma-glaucoma-expert or change ADAPTER_DIR."
    )

ADAPTER_DIR = find_adapter_dir()

print("Using adapter folder:", ADAPTER_DIR)
print("Files:")
for f in sorted(os.listdir(ADAPTER_DIR)):
    print(" -", f)


Using adapter folder: /content/medgemma-glaucoma-expert
Files:
 - adapter_config.json
 - adapter_model.safetensors
 - chat_template.jinja
 - processor_config.json
 - tokenizer.json
 - tokenizer_config.json


## Load base MedGemma + saved LoRA adapter

In [ ]:
import re
import threading

In [ ]:
BASE_MODEL_ID = "google/medgemma-1.5-4b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# Prefer the saved processor because it was exported with the adapter.
processor = AutoProcessor.from_pretrained(ADAPTER_DIR)

base_model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

print("Loaded base model:", BASE_MODEL_ID)
print("Loaded LoRA adapter:", ADAPTER_DIR)
print("Model device:", getattr(model, "device", next(model.parameters()).device))

## Inference functions

In [ ]:
CLASSIFICATION_PROMPT = "Does this retinal image show signs of Glaucoma? Answer yes or no."


def move_inputs_to_model_device(inputs):
    # Move tensor inputs to the active model device for single-GPU Kaggle inference.
    device = getattr(model, "device", None)
    if device is None:
        device = next(model.parameters()).device
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in inputs.items()}


def build_image_message(image, prompt):
    return [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]


def run_logit_diagnosis(image_path):
    image = Image.open(image_path).convert("RGB")
    msg = build_image_message(image, CLASSIFICATION_PROMPT)

    inputs = processor.apply_chat_template(
        msg,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = move_inputs_to_model_device(inputs)

    with torch.no_grad():
        outputs = model(**inputs)
        next_token_logits = outputs.logits[:, -1, :]

    # Using encode is safer than convert_tokens_to_ids because the tokenizer may split text.
    yes_id = processor.tokenizer.encode("yes", add_special_tokens=False)[-1]
    no_id = processor.tokenizer.encode("no", add_special_tokens=False)[-1]

    yes_no_logits = torch.stack([
        next_token_logits[0, yes_id],
        next_token_logits[0, no_id],
    ])

    yes_prob, no_prob = F.softmax(yes_no_logits, dim=0).detach().float().cpu().tolist()

    if yes_prob > no_prob:
        prediction = "yes"
        confidence = yes_prob
    else:
        prediction = "no"
        confidence = no_prob

    return {
        "prediction": prediction,
        "confidence": confidence,
        "yes_probability": yes_prob,
        "no_probability": no_prob,
    }

In [ ]:
def generate_glaucoma_explanation(image_path, diagnosis=None, max_new_tokens=1000):
    image = Image.open(image_path).convert("RGB")

    if diagnosis is None:
        diagnosis = run_logit_diagnosis(image_path)

    pred = diagnosis["prediction"]
    conf = diagnosis["confidence"]
    yes_prob = diagnosis["yes_probability"]
    no_prob = diagnosis["no_probability"]

    verdict_text = "likely glaucoma / GON+" if pred == "yes" else "likely non-glaucoma / GON-"

    explanation_prompt = f"""
You are assisting with ophthalmology AI research.

A separate yes/no classifier using this same retinal fundus image predicted: {pred}.
Interpreted label: {verdict_text}.
Relative confidence among yes/no answers: {conf:.2%}.
Yes probability: {yes_prob:.2%}.
No probability: {no_prob:.2%}.

Look at the image and write a concise possible visual explanation for why the classifier may have predicted this.

Important safety constraints:
- Do not claim certainty.
- Do not invent findings that are not visible.
- Do not present this as a clinical diagnosis.
- Focus on visible glaucoma-related fundus features only.
- Mention uncertainty if image quality, optic disc visibility, or cup/disc boundary is limited.

Maximum 5 sentences per summary. Max 5 points each per Possible visual evidence and Features reducing confidence.
Limit your reasoning to 4 compact points.

Use this exact format:

Possible visual evidence:
- ...

Features reducing confidence:
- ...

Summary:
...
""".strip()

    msg = build_image_message(image, explanation_prompt)

    inputs = processor.apply_chat_template(
        msg,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = move_inputs_to_model_device(inputs)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    input_len = inputs["input_ids"].shape[-1]
    generated_text = processor.tokenizer.decode(
        generated_ids[0][input_len:],
        skip_special_tokens=True,
    ).strip()

    return {
        **diagnosis,
        "explanation": generated_text,
    }

## XAI heatmap function

This converts the original notebook's occlusion heatmap into a reusable function for uploaded images.


In [ ]:
# Batched XAI heatmap for the exact uploaded image.
# This uses occlusion sensitivity: mask one grid patch at a time and see how much
# the model's confidence for its predicted class drops.
#
# Main speedup:
# Instead of running one model forward per occluded patch, we batch the occluded
# images and run them together on the GPU.
#
# For XAI_GRID_SIZE = 4:
#   old path: 16 separate model forwards
#   new path with XAI_BATCH_SIZE = 16: 1 batched model forward
#
# If you get CUDA OOM, lower XAI_BATCH_SIZE to 8, 4, or 2.

XAI_BATCH_SIZE = 16

YES_ID = processor.tokenizer.encode("yes", add_special_tokens=False)[-1]
NO_ID = processor.tokenizer.encode("no", add_special_tokens=False)[-1]


def _last_nonpad_token_indices(attention_mask):
    """
    Returns the index of the last non-padding token for each row.
    Works for both left-padding and right-padding.
    """
    return (
        attention_mask.shape[1]
        - 1
        - attention_mask.long().flip(dims=[1]).argmax(dim=1)
    )


def run_logit_diagnosis_batch_pil(images, batch_size=XAI_BATCH_SIZE):
    """
    Batched version of run_logit_diagnosis for PIL images.

    Returns a list of dicts with the same keys as run_logit_diagnosis:
      prediction, confidence, yes_probability, no_probability
    """
    if not images:
        return []

    results = []
    n = len(images)
    i = 0

    while i < n:
        current_batch_size = min(batch_size, n - i)
        batch_images = [img.convert("RGB") for img in images[i:i + current_batch_size]]

        conversations = [
            build_image_message(img, CLASSIFICATION_PROMPT)
            for img in batch_images
        ]

        try:
            inputs = processor.apply_chat_template(
                conversations,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                padding=True,
            )

            inputs = move_inputs_to_model_device(inputs)

            with torch.inference_mode():
                outputs = model(**inputs)
                logits = outputs.logits

                if "attention_mask" in inputs:
                    last_indices = _last_nonpad_token_indices(inputs["attention_mask"])
                    last_indices = last_indices.to(logits.device)
                else:
                    last_indices = torch.full(
                        (logits.shape[0],),
                        logits.shape[1] - 1,
                        dtype=torch.long,
                        device=logits.device,
                    )

                batch_indices = torch.arange(logits.shape[0], device=logits.device)
                next_token_logits = logits[batch_indices, last_indices, :]

                yes_no_logits = torch.stack(
                    [
                        next_token_logits[:, YES_ID],
                        next_token_logits[:, NO_ID],
                    ],
                    dim=1,
                )

                probs = F.softmax(yes_no_logits.float(), dim=1).detach().cpu().tolist()

            for yes_prob, no_prob in probs:
                if yes_prob > no_prob:
                    prediction = "yes"
                    confidence = yes_prob
                else:
                    prediction = "no"
                    confidence = no_prob

                results.append({
                    "prediction": prediction,
                    "confidence": confidence,
                    "yes_probability": float(yes_prob),
                    "no_probability": float(no_prob),
                })

            i += current_batch_size

        except torch.cuda.OutOfMemoryError:
            if batch_size <= 1:
                raise

            torch.cuda.empty_cache()
            batch_size = max(1, batch_size // 2)
            print(f"CUDA OOM during batched XAI. Retrying with XAI batch size {batch_size}.")

    return results


def make_xai_heatmap_data_url(image_path, base_result=None, grid_size=4, batch_size=XAI_BATCH_SIZE):
    if base_result is None:
        base_result = run_logit_diagnosis(image_path)

    img = Image.open(image_path).convert("RGB")
    W, H = img.size

    predicted_class = base_result["prediction"]
    target_key = "yes_probability" if predicted_class == "yes" else "no_probability"
    base_target_prob = base_result[target_key]

    heatmap = np.zeros((grid_size, grid_size), dtype=np.float32)
    patch_w = max(1, W // grid_size)
    patch_h = max(1, H // grid_size)
    mask_color = (128, 128, 128)

    occluded_images = []
    occluded_positions = []

    for gy in range(grid_size):
        for gx in range(grid_size):
            masked = img.copy()
            draw = ImageDraw.Draw(masked)

            x0 = gx * patch_w
            y0 = gy * patch_h
            x1 = W if gx == grid_size - 1 else (gx + 1) * patch_w
            y1 = H if gy == grid_size - 1 else (gy + 1) * patch_h

            draw.rectangle([x0, y0, x1, y1], fill=mask_color)

            occluded_images.append(masked)
            occluded_positions.append((gy, gx))

    masked_results = run_logit_diagnosis_batch_pil(
        occluded_images,
        batch_size=batch_size,
    )

    for (gy, gx), masked_result in zip(occluded_positions, masked_results):
        masked_target_prob = masked_result[target_key]
        heatmap[gy, gx] = base_target_prob - masked_target_prob

    # Normalize only positive evidence for the predicted class.
    heatmap = np.maximum(heatmap, 0)

    if float(heatmap.max()) > 0:
        heatmap_norm = heatmap / float(heatmap.max())
    else:
        heatmap_norm = heatmap

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(img)
    ax.imshow(
        heatmap_norm,
        cmap="jet",
        alpha=0.45,
        interpolation="bilinear",
        extent=(0, W, H, 0),
    )
    ax.axis("off")
    ax.set_title(f"XAI occlusion heatmap: predicted {predicted_class}")

    buffer = io.BytesIO()
    plt.savefig(buffer, format="png", bbox_inches="tight", pad_inches=0.05, dpi=150)
    plt.close(fig)
    buffer.seek(0)

    encoded = base64.b64encode(buffer.read()).decode("utf-8")
    return "data:image/png;base64," + encoded

## Start website upload API

Run this last. It blocks while the API is alive. Keep the Colab runtime running.

In [ ]:
# Flask API for website image uploads.
# Run this cell LAST. It prints a public URL.
#
# Your website should POST an image file named "image" to:
#   <PUBLIC_URL>/process

NGROK_AUTHTOKEN = "Insert your token"
API_KEY = "Insert your key"


# 4 is faster. Use 6 or 8 for a sharper but slower heatmap.
XAI_GRID_SIZE = 4

# Prevent overlapping requests from double-clicks or browser retries.
# Colab + one GPU is much more stable when requests run one at a time.
inference_lock = threading.Lock()

ngrok.set_auth_token(NGROK_AUTHTOKEN)

app = Flask(__name__)
app.config["MAX_CONTENT_LENGTH"] = 12 * 1024 * 1024
CORS(app)


@app.route("/", methods=["GET"])
def home():
    return jsonify({
        "status": "ok",
        "endpoint": "/process",
        "usage": "POST multipart/form-data with image=<file>",
    })


@app.route("/process", methods=["POST"])
def process_image():
    if request.headers.get("X-API-Key") != API_KEY:
        return jsonify({"error": "unauthorized"}), 401

    if "image" not in request.files:
        return jsonify({"error": "No image field found. Send multipart form field named image."}), 400

    uploaded = request.files["image"]

    if uploaded.filename == "":
        return jsonify({"error": "Empty filename."}), 400

    safe_name = secure_filename(uploaded.filename)
    ext = os.path.splitext(safe_name)[1].lower()

    if ext not in [".jpg", ".jpeg", ".png", ".webp", ".bmp"]:
        ext = ".jpg"

    temp_path = f"/tmp/upload_{uuid.uuid4().hex}{ext}"
    uploaded.save(temp_path)

    try:
        with inference_lock:
            diagnosis = run_logit_diagnosis(temp_path)

            explanation_result = generate_glaucoma_explanation(
                temp_path,
                diagnosis=diagnosis,
            )

            heatmap_data_url = make_xai_heatmap_data_url(
                temp_path,
                base_result=diagnosis,
                grid_size=XAI_GRID_SIZE,
            )
            clean = re.sub(r"(?s).*?(?:<unused95>|answer)", "", explanation_result["explanation"]).strip()

        return jsonify({
            "prediction": explanation_result["prediction"],
            "confidence": explanation_result["confidence"],
            "yes_probability": explanation_result["yes_probability"],
            "no_probability": explanation_result["no_probability"],
            "explanation": clean if clean else explanation_result["explanation"].split("Summary:")[-1].strip(),
            "xai_heatmap": heatmap_data_url,
            "xai_type": "occlusion_sensitivity",
            "xai_grid_size": XAI_GRID_SIZE,
            "xai_note": "Heatmap highlights regions where masking reduced confidence in the predicted class. This is research/demo XAI, not clinical evidence.",
        })

    except Exception as e:
        return jsonify({"error": str(e)}), 500

    finally:
        try:
            os.remove(temp_path)
        except Exception:
            pass

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


public_url = ngrok.connect(5000).public_url

print("PUBLIC URL:", public_url)
print("PROCESS ENDPOINT:", public_url + "/process")

app.run(host="0.0.0.0", port=5000, threaded=False)